
## Spark Window Functions in PySpark

### What are Window Functions?

Spark **Window Functions** allow you to perform calculations across a group of rows related to the current row **without reducing the number of rows**.

### Difference Between `groupBy()` and Window Functions

| groupBy() | Window Function |
|------------|-----------------|
| Aggregates rows | Keeps all rows |
| Returns one row per group | Returns one row for every input row |
| Used for summary reports | Used for ranking, running totals, comparisons |

---

### Sample Dataset

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

data = [
    ("John", "IT", 5000),
    ("David", "IT", 7000),
    ("Alex", "IT", 6000),
    ("Mary", "HR", 4000),
    ("Lisa", "HR", 4500)
]

df = spark.createDataFrame(data, ["Employee", "Department", "Salary"])
df.show()
```

Output

```
+--------+----------+------+
|Employee|Department|Salary|
+--------+----------+------+
|John    |IT        |5000  |
|David   |IT        |7000  |
|Alex    |IT        |6000  |
|Mary    |HR        |4000  |
|Lisa    |HR        |4500  |
+--------+----------+------+
```

---

### Creating a Window Specification

```python
from pyspark.sql.window import Window

windowSpec = Window.partitionBy("Department").orderBy("Salary")
```

- **partitionBy()** → Groups rows.
- **orderBy()** → Sorts rows within each partition.

---

### 1. ROW_NUMBER()

Assigns a unique sequential number to each row.

```python
from pyspark.sql.functions import row_number, desc

windowSpec = Window.partitionBy("Department").orderBy(desc("Salary"))

df.withColumn(
    "Row_Number",
    row_number().over(windowSpec)
).show()
```

Output

|Department|Salary|Row_Number|
|----------|------|----------|
|IT|7000|1|
|IT|6000|2|
|IT|5000|3|
|HR|4500|1|
|HR|4000|2|

### Use Cases

- Top N records
- Remove duplicates
- Pagination

---

### 2. RANK()

Assigns the same rank for duplicate values but skips the next rank.

Example

|Salary|Rank|
|------|----|
|7000|1|
|7000|1|
|6000|3|

```python
from pyspark.sql.functions import rank

df.withColumn(
    "Rank",
    rank().over(windowSpec)
).show()
```

---

### 3. DENSE_RANK()

Assigns the same rank for duplicate values without skipping ranks.

Example

|Salary|Dense Rank|
|------|----------|
|7000|1|
|7000|1|
|6000|2|

```python
from pyspark.sql.functions import dense_rank

df.withColumn(
    "DenseRank",
    dense_rank().over(windowSpec)
).show()
```

---

### ROW_NUMBER vs RANK vs DENSE_RANK

|Salary|ROW_NUMBER|RANK|DENSE_RANK|
|------|----------|----|----------|
|7000|1|1|1|
|7000|2|1|1|
|6000|3|3|2|
|5000|4|4|3|

---

### 4. LEAD()

Returns the next row value.

```python
from pyspark.sql.functions import lead

windowSpec = Window.orderBy("Salary")

df.withColumn(
    "NextSalary",
    lead("Salary").over(windowSpec)
).show()
```

Output

|Salary|NextSalary|
|------|----------|
|4000|4500|
|4500|5000|
|5000|6000|
|6000|7000|
|7000|NULL|

### Use Cases

- Future comparisons
- Forecasting
- Trend analysis

---

### 5. LAG()

Returns the previous row value.

```python
from pyspark.sql.functions import lag

df.withColumn(
    "PreviousSalary",
    lag("Salary").over(windowSpec)
).show()
```

Output

|Salary|PreviousSalary|
|------|--------------|
|4000|NULL|
|4500|4000|
|5000|4500|
|6000|5000|
|7000|6000|

### Example

```python
from pyspark.sql.functions import col

df.withColumn(
    "PreviousSalary",
    lag("Salary").over(windowSpec)
).withColumn(
    "Difference",
    col("Salary") - col("PreviousSalary")
).show()
```

---

### 6. FIRST()

Returns the first value in each partition.

```python
from pyspark.sql.functions import first

windowSpec = Window.partitionBy("Department").orderBy("Salary")

df.withColumn(
    "LowestSalary",
    first("Salary").over(windowSpec)
).show()
```

Output

|Department|LowestSalary|
|----------|------------|
|IT|5000|
|HR|4000|

---

### 7. LAST()

Returns the last value in each partition.

```python
from pyspark.sql.functions import last

df.withColumn(
    "HighestSalary",
    last("Salary").over(
        windowSpec.rowsBetween(
            Window.unboundedPreceding,
            Window.unboundedFollowing
        )
    )
).show()
```

Output

|Department|HighestSalary|
|----------|-------------|
|IT|7000|
|HR|4500|

---

### 8. SUM()

Calculates running totals.

```python
from pyspark.sql.functions import sum

windowSpec = Window.partitionBy("Department").orderBy("Salary")

df.withColumn(
    "RunningTotal",
    sum("Salary").over(windowSpec)
).show()
```

Output

|Salary|RunningTotal|
|------|------------|
|5000|5000|
|6000|11000|
|7000|18000|

---

### 9. AVG()

Calculates the average within a partition.

```python
from pyspark.sql.functions import avg

df.withColumn(
    "AverageSalary",
    avg("Salary").over(
        Window.partitionBy("Department")
    )
).show()
```

Output

|Department|AverageSalary|
|----------|-------------|
|IT|6000|
|HR|4250|

---

### 10. MIN()

Returns the minimum value.

```python
from pyspark.sql.functions import min

df.withColumn(
    "MinSalary",
    min("Salary").over(
        Window.partitionBy("Department")
    )
).show()
```

Output

|Department|MinSalary|
|----------|---------|
|IT|5000|
|HR|4000|

---

### 11. MAX()

Returns the maximum value.

```python
from pyspark.sql.functions import max

df.withColumn(
    "MaxSalary",
    max("Salary").over(
        Window.partitionBy("Department")
    )
).show()
```

Output

|Department|MaxSalary|
|----------|---------|
|IT|7000|
|HR|4500|

---

### 12. NTILE()

Divides rows into equal buckets.

```python
from pyspark.sql.functions import ntile

windowSpec = Window.orderBy("Salary")

df.withColumn(
    "Bucket",
    ntile(2).over(windowSpec)
).show()
```

Output

|Salary|Bucket|
|------|------|
|4000|1|
|4500|1|
|5000|1|
|6000|2|
|7000|2|

### Use Cases

- Quartiles
- Percentiles
- Customer segmentation

---

## Window Frame

A window frame controls which rows are included in the calculation.

```python
windowSpec = Window.partitionBy("Department") \
    .orderBy("Salary") \
    .rowsBetween(-1, 1)
```

This frame includes:

- Previous row
- Current row
- Next row

Example

|Salary|
|------|
|5000|
|6000|
|7000|

For salary **6000**, the frame contains:

```
5000
6000
7000
```

---

## Real-Time Interview Scenarios

### 1. Highest Paid Employee in Each Department

```python
windowSpec = Window.partitionBy("Department").orderBy(desc("Salary"))

df.withColumn(
    "rn",
    row_number().over(windowSpec)
).filter("rn = 1")
```

---

### 2. Second Highest Salary

```python
windowSpec = Window.orderBy(desc("Salary"))

df.withColumn(
    "rank",
    dense_rank().over(windowSpec)
).filter("rank = 2")
```

---

### 3. Remove Duplicate Records

```python
windowSpec = Window.partitionBy("Employee").orderBy(desc("Salary"))

df.withColumn(
    "rn",
    row_number().over(windowSpec)
).filter("rn = 1")
```

---

### 4. Month-over-Month Sales Comparison

```python
windowSpec = Window.partitionBy("Product").orderBy("Month")

df.withColumn(
    "PreviousMonthSales",
    lag("Sales").over(windowSpec)
).withColumn(
    "Growth",
    col("Sales") - col("PreviousMonthSales")
)
```

---

## Summary

| Function | Purpose | Common Use Case |
|----------|---------|-----------------|
| `row_number()` | Unique numbering | Remove duplicates, Top-N |
| `rank()` | Ranking with gaps | Competition ranking |
| `dense_rank()` | Ranking without gaps | Top-N reporting |
| `lead()` | Next row value | Forecasting |
| `lag()` | Previous row value | Trend analysis |
| `first()` | First value in partition | Earliest or lowest value |
| `last()` | Last value in partition | Latest or highest value |
| `sum()` | Running total | Cumulative calculations |
| `avg()` | Average | Department average |
| `min()` | Minimum value | Lowest value |
| `max()` | Maximum value | Highest value |
| `ntile()` | Divide rows into buckets | Quartiles, segmentation |

---

## Key Interview Points

- Window functions **do not reduce the number of rows**.
- `partitionBy()` defines the logical group.
- `orderBy()` determines row order within each partition.
- `row_number()` always produces unique sequence numbers.
- `rank()` skips numbers after ties.
- `dense_rank()` does not skip numbers after ties.
- `lag()` looks at previous rows.
- `lead()` looks at next rows.
- Aggregation functions (`sum`, `avg`, `min`, `max`) can also be used as window functions.
- Window frames (`rowsBetween` or `rangeBetween`) allow fine-grained control over which rows participate in the calculation.

In [0]:
data = [
    ("John", "IT", 5000, "2024-01", 10),
    ("David", "IT", 7000, "2024-02", 20),
    ("Alex", "IT", 6000, "2024-03", 15),
    ("Mary", "HR", 4000, "2024-01", 12),
    ("Lisa", "HR", 4500, "2024-02", 18),
    ("Sam", "HR", 4500, "2024-03", 18),
    ("Tom", "Finance", 8000, "2024-01", 25),
    ("Anna", "Finance", 9000, "2024-02", 30),
    ("Eve", "Finance", 8500, "2024-03", 28)
]

df = spark.createDataFrame(
    data,
    ["Employee", "Department", "Salary", "Month", "Sales"]
)

display(df)

In [0]:
from pyspark.sql.window import Window

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

windowspec=Window.partitionBy("Department").orderBy(col("Salary").desc())

In [0]:
df_row_num=df.withColumn("row_num",row_number().over(windowspec))\
    .withColumn("danse_rnk",dense_rank().over(windowspec))\
        .withColumn("rnk",rank().over(windowspec))
display(df_row_num )

In [0]:
ws=Window.orderBy(col("Salary").desc())

df_max=df.withColumn("max1",max(col("Salary")).over(windowspec))\
    .withColumn("max2",max(col("Salary")).over(ws))

display(df_max)

In [0]:
df.withColumn(
    "MaxSalary",
    max("Salary").over(
        Window.partitionBy("Department")
    )
).show()

In [0]:
df.groupBy("Department").agg(max("Salary").alias("MaxSalary")).show()

In [0]:
df.groupBy("Department").agg(
    count("*").alias("Employees"),
    sum("Salary").alias("TotalSalary"),
    avg("Salary").alias("AverageSalary"),
    max("Salary").alias("HighestSalary"),
    min("Salary").alias("LowestSalary")
).show()